# Test Epic_medical_history


In [ ]:
import os
import random
import shutil
import sys

import numpy as np

In [ ]:
random_seed_value = 42

np.random.seed(random_seed_value)
random.seed(random_seed_value)

In [ ]:
current_dir = os.getcwd()
grandparent_dir = os.path.dirname(os.path.dirname(current_dir))

sys.path.insert(0, os.path.join(grandparent_dir, "pat2vec"))
sys.path.append(grandparent_dir)
pat2vec_dir = os.path.abspath(os.path.join(grandparent_dir, "pat2vec"))
print(f"Pat2vec path: {pat2vec_dir}")

In [ ]:
TEMP_BASE_DIR = "/tmp/epic_medical_history_test_project"

try:
    shutil.rmtree(TEMP_BASE_DIR, ignore_errors=True)
except Exception as e:
    msg = f"Failed to clean up directory: {e}"
    raise RuntimeError(msg)
print("Previous outputs cleaned.")

In [ ]:
from nb_temp_setup import get_notebook_port_offsetfrom pat2vec.util.docker_elastic import ElasticContainerport_offset = get_notebook_port_offset()es_container = ElasticContainer(port_offset=port_offset)es_container.stop()print("Starting Elasticsearch container...")if not es_container.start():    msg = "Failed to start Elasticsearch."    raise RuntimeError(msg)host, username, password = es_container.get_credentials()creds_filename = "test_elastic_credentials_epic_medical_history_get.py"creds_content = f"""\nusername = '{username}\'\npassword = '{password}\'\napi_key = None\nhosts = ["{host}"]\n"""with open(creds_filename, "w") as f:    f.write(creds_content)print(f"Created {creds_filename}")

In [ ]:
from pat2vec.util.config_pat2vec import config_class

schema_path = "test_files/elastic_schemas.json"

config_populate = config_class(
    proj_name=f"{TEMP_BASE_DIR}",
    credentials_path=creds_filename,
    test_schema_path=schema_path,
    testing=True,
    testing_elastic=True,
    global_start_year=2020,
    global_start_month=1,
    global_start_day=1,
    global_end_year=2023,
    global_end_month=12,
    global_end_day=31,
)

In [ ]:
from pat2vec.util.get_dummy_data_cohort_searcher import populate_elastic_with_dummy_data

print("Populating test Elasticsearch cluster with dummy data...")
patient_ids = populate_elastic_with_dummy_data(config_populate, n_patients=5)
print(f"Population complete. Generated {len(patient_ids)} dummy patients.")

In [ ]:
from pat2vec.pat2vec_search.cogstack_search_methods import initialize_cogstack_client

cs = initialize_cogstack_client(config_populate)

indices = ["epr_documents", "basic_observations", "observations", "order", "pims_apps"]
print("Refreshing indices...")
cs.elastic.indices.refresh(index=indices, ignore_unavailable=True)
import time

time.sleep(2)
print("Indices refreshed.")

In [ ]:
PROJ_NAME = f"{TEMP_BASE_DIR}"
DB_FILENAME = "temp_epic_medical_history_db.sqlite"
DB_PATH = os.path.join(PROJ_NAME, "outputs", DB_FILENAME)

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

try:
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
except Exception as e:
    msg = f"Failed to remove database: {e}"
    raise RuntimeError(msg)

db_connection_string = "sqlite:///" + DB_PATH
print(f"Database connection string set to: {db_connection_string}")

In [ ]:
from pat2vec.util.logger_setup import setup_logger

logger = setup_logger()
print("Logger initialized.")

In [ ]:
from pat2vec.main_pat2vec import main

config_obj = config_class(
    proj_name=PROJ_NAME,
    credentials_path=creds_filename,
    current_path_dir="",
    main_options={"epic_medical_history": True},
    batch_mode=True,
    verbosity=0,
    random_seed_val=random_seed_value,
    testing=True,
    testing_elastic=True,
    dummy_medcat_model=True,
    use_controls=False,
    medcat=False,
    start_time=None,
    patient_id_column_name="client_idcode",
    annot_filter_options={},
    shuffle_pat_list=False,
    storage_backend="database",
    db_connection_string=db_connection_string,
    check_patient_existence=False,
    treatment_doc_filename="test_files/treatment_docs.csv",
)
pat2vec_obj = main(
    cogstack=True,
    use_filter=False,
    json_filter_path=None,
    random_seed_val=random_seed_value,
    hostname=None,
    config_obj=config_obj,
)

In [ ]:
print("\n=== PROCESSING PATIENTS WITH pat_maker ===")
print(f"Patient list: {pat2vec_obj.all_patient_list}")

try:
    print(f"Processing patient 0: {pat2vec_obj.all_patient_list[0]}")
    pat2vec_obj.pat_maker(0)
except Exception as e:
    msg = f"Failed to process patient 0 with pat_maker: {e}. Critical error - pipeline failed."
    raise RuntimeError(
        msg,
    ) from e
print("Patient feature extraction complete.")

In [ ]:
from pat2vec.util.helper_functions import get_all_features

all_features = get_all_features(config_obj)

if all_features.empty:
    msg = "FATAL ERROR: get_all_features returned empty DataFrame"
    raise RuntimeError(msg)
print(f"Successfully retrieved {len(all_features)} rows from database.")

In [ ]:
# === VECTOR VALIDATION ===
feature_cols = [c for c in all_features.columns if c.startswith("med_hist_")]

assert len(feature_cols) > 0, "No feature columns found. Available columns: " + str(
    list(all_features.columns)
)

non_null_counts = all_features[feature_cols].notna().sum()
totally_empty = non_null_counts[non_null_counts == 0]

assert len(totally_empty) == 0, (
    f"The following feature columns are entirely null:\n"
    f"{list(totally_empty.index)}\n"
    "Vectorisation is silently failing — check the get method return value."
)

print("Feature columns (" + str(len(feature_cols)) + "): " + str(feature_cols))
print("Non-null counts per feature column:")
for col in sorted(feature_cols):
    val = all_features[col].notna().sum()
    print("  " + str(col) + ": " + str(val) + " non-null values")

In [ ]:
from pat2vec.util.post_processing_build_methods import (
    build_merged_epr_mct_annot_df,
    build_merged_epr_mct_doc_df,
)

In [ ]:
print("\n=== DATABASE AND PROJECT CLEANUP ===")

try:
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
        print(f"Removed database: {DB_PATH}")
except Exception as e:
    msg = f"Failed to remove {DB_PATH}: {e}"
    raise RuntimeError(msg)

try:
    if os.path.exists(TEMP_BASE_DIR):
        shutil.rmtree(TEMP_BASE_DIR, ignore_errors=False)
        print(f"Removed project directory: {TEMP_BASE_DIR}")
except Exception as e:
    msg = f"Failed to remove {TEMP_BASE_DIR}: {e}"
    raise RuntimeError(msg)

try:
    if os.path.exists(creds_filename):
        os.remove(creds_filename)
        print(f"Removed credentials: {creds_filename}")
except Exception as e:
    msg = f"Failed to remove {creds_filename}: {e}"
    raise RuntimeError(msg)

In [ ]:
from pat2vec.util.post_processing_build_methods import (
    build_merged_epr_mct_annot_df,
    build_merged_epr_mct_doc_df,
)

In [ ]:
print("\n=== FINAL VERIFICATION ===")
assert not os.path.exists(DB_PATH), "Database file still exists!"
assert not os.path.exists(TEMP_BASE_DIR), "Project directory still exists!"
assert not os.path.exists(creds_filename), "Credentials file still exists!"

print("All cleanup verified - no residual files remain.")
print("\n=== TEST SUCCESSFUL ===")